In [2]:
import os
import json
import glob
import re
import pandas as pd
from rouge_score import rouge_scorer
from tqdm import tqdm

In [ ]:
def clean_text(text):
    """
    清洗逻辑：
    1. 移除 <think>...</think> (包括跨行内容)
    2. 移除 [PROVE:...] 标签
    3. 移除 /no_think
    4. 规范化空白字符
    """
    if not isinstance(text, str):
        return ""
    

    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
    

    text = re.sub(r'\[PROVE:.*?\]', '', text, flags=re.DOTALL)
    
  
    text = text.replace('/no_think', '')
    

    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

In [ ]:
def main():
    print(f"--- 开始处理 ---")
    print(f"输入目录: {INPUT_DIR}")
    print(f"输出目录: {OUTPUT_DIR}")
    os.makedirs(OUTPUT_DIR, exist_ok=True)


    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

 
    files = glob.glob(os.path.join(INPUT_DIR, '*.jsonl'))

    if not files:
        print("错误: 未找到数据文件。")
        return

    all_summaries = []

    for filepath in tqdm(files, desc="处理文件"):
        filename = os.path.basename(filepath)
        output_filepath = os.path.join(OUTPUT_DIR, f"scored_{filename}")
        
        processed_count = 0
        file_scores = [] 
        
        
        with open(filepath, 'r', encoding='utf-8') as f_in, \
             open(output_filepath, 'w', encoding='utf-8') as f_out:
            
            for line in f_in:
                try:
                    line = line.strip()
                    if not line: continue
                    
                    item = json.loads(line)
                    
                  
                    if 'response' not in item or 'labels' not in item:
                        continue

                  
                    raw_response = item['response']
                    raw_labels = item['labels']
                    
                    clean_res = clean_text(raw_response)
                    clean_lab = clean_text(raw_labels)

                    if not clean_res or not clean_lab:
                        rouge_l = 0.0
                    else:
                        scores = scorer.score(clean_lab, clean_res)
                        rouge_l = scores['rougeL'].fmeasure

                    
                    item['cleaned_response'] = clean_res
                    item['cleaned_labels'] = clean_lab
                    item['rougeL_score'] = rouge_l
                    
                  
                    f_out.write(json.dumps(item, ensure_ascii=False) + '\n')
                    
                    file_scores.append(rouge_l)
                    processed_count += 1

                except json.JSONDecodeError:
                    continue
        
      
        if file_scores:
            avg_score = sum(file_scores) / len(file_scores)
            all_summaries.append({
                'filename': filename,
                'count': processed_count,
                'avg_rougeL': avg_score
            })
          

    print("\n--- 处理完成，生成汇总 ---")
    if all_summaries:
        df = pd.DataFrame(all_summaries)
        csv_path = os.path.join(OUTPUT_DIR, 'summary_rouge_scores.csv')
        df.to_csv(csv_path, index=False)
        print(f"汇总 CSV 已保存: {csv_path}")
        print(df.to_string())
    else:
        print("未处理任何有效数据。")

In [ ]:
# ================= 配置区域 =================
INPUT_DIR = ''               # 原始数据集文件夹
OUTPUT_DIR = './rouge_results'     # 结果保存文件夹
# ===========================================

In [21]:
if __name__ == "__main__":
    main()

--- 开始处理 ---
输入目录: /usr/data/wjx/trove/eval/other_models_infer_data/
输出目录: ./rouge_results


处理文件: 100%|██████████| 1/1 [00:00<00:00,  2.12it/s]


--- 处理完成，生成汇总 ---
汇总 CSV 已保存: ./rouge_results/summary_rouge_scores.csv
                           filename  count  avg_rougeL
0  valid_gpt5_infer_formatted.jsonl    279    0.412503
